In [ ]:
import pandas as pd
import folium
import base64
from IPython.display import IFrame, display

In [ ]:
# Constants
# CSS for the circle markers to be used in the legend
marker_css = """
.legend-labels span {
    float: left;
    margin-right: 8px;
    margin-top: 3px;
    border-radius: 50%;
}

/* Small circle */
.legend-labels span.small {
    height: 6px;
    width: 6px;
}

/* Medium circle */
.legend-labels span.medium {
    height: 9px;
    width: 9px;
}

/* Large circle */
.legend-labels span.large {
    height: 15px;
    width: 15px;
}
</style>
"""

In [ ]:
# Read the CSV file containing all information 
fileName = '../../../../../data/Japan_Meteorological_Agency/stations/amedas_df_all.csv'
amedas_df_all = pd.read_csv(fileName, encoding='utf-8')

In [ ]:
# Make a dictionary to be used when styling the jurisdiction maps
juris_info = {
    'Sapporo': ['#EF476F', (11, 24)],
    'Sendai': ['#F78C6B', (31, 36)],
    'Tokyo': ['#FFD166', (40, 57)],
    'Osaka': ['#06D6A0', (60, 74)],
    'Fukuoka': ['#118AB2', (81, 88)],
    'Okinawa': ['#073B4C', (91, 94)],
}

# Work through each jurisdiction to find the best central coordinates to center the map
for key, val in juris_info.items():
    prec_low = val[1][0]
    prec_high = val[1][1]
    
    # We only want the latitudes and longitudes pertaining to the jurisdiction
    lats = [
        float(rowNow.latitude_decimal)
        for rowNow in amedas_df_all.itertuples()
        if prec_low <= int(rowNow.prec_no) <= prec_high
    ]
    longs = [
        float(rowNow.longitude_decimal)
        for rowNow in amedas_df_all.itertuples()
        if prec_low <= int(rowNow.prec_no) <= prec_high
    ]
    
    # Max and min of latitude and longitudes
    min_lat = min(lats)
    max_lat = max(lats)
    min_long = min(longs)
    max_long = max(longs)

    # Center point + some padding on right hand side
    mid_lat = min_lat + ((max_lat - min_lat) / 2)
    mid_long = min_long + ((max_long - min_long) / 2) + (max_long - min_long) * 0.3

    # Append the location as tuple to the dictionary entry
    juris_info[key].append((mid_lat, mid_long))

In [ ]:
def ColorHex2RGB(hex_str):
    """Function to convert the hexadecimal color to RGB notation
    AUTHOR:     Mai Tanaka (www.DataDrivenMai.com)
    DATE:       2026-06-30
    REQUIRES:   hex_str = hexadecimal notation of the color (eg. '#06D6A0')
    RETURNS:    rgb_val = RGB values in a list (eg. [6, 214, 160])
    """
    # Remove the hash symbol 
    hex_str = hex_str.lstrip('#')
    
    # Convert hex pairs to integers
    rgb_val = []
    for i in (0, 2, 4):
        rgb_val.append(int(hex_str[i:i+2], 16))
        
    return rgb_val

In [ ]:
def PrecNo2Name(arg_prec_no):
    """Function convert a prefecture number to a jurisdiction name
    AUTHOR:     Mai Tanaka (www.DataDrivenMai.com)
    DATE:       2026-06-30
    REQUIRES:   arg_prec_no = prefectural number (eg. 24)
    RETURNS:    juris_now = name of jurisdiction (eg. 'Sapporo')
    """
    # Simple if-else to get the appropriate jurisdiction
    if arg_prec_no <= 24:
        juris_now = 'Sapporo'
    elif arg_prec_no <= 36:
        juris_now = 'Sendai'
    elif arg_prec_no <= 57:
        juris_now = 'Tokyo'
    elif arg_prec_no <= 74:
        juris_now = 'Osaka'
    elif arg_prec_no <= 88:
        juris_now = 'Fukuoka'
    elif arg_prec_no <= 94:
        juris_now = 'Okinawa'

    return juris_now

In [ ]:
def CreateLegend4CircleMarkers(legend_title, arg_width, arg_legend_list):
    """Function to create a legend for circle markers 
    AUTHOR:     Mai Tanaka (www.DataDrivenMai.com)
    DATE:       2026-06-30
    REQUIRES:   legend_title = title of the legend as a string
                arg_width = legend width in pixels
                arg_legend_list = legend entries as a list, with each entry containing 
                                [marker_descrip_text, marker_html_class, 
                                 marker_stroke_color, stroke_opacity, 
                                 marker_fill_color, fill_opacity]
    RETURNS:    legend_html used to generate legend with m.get_root().html.add_child(folium.Element(legend_html))
    """
    # Basic CSS for the legend
    basic_CSS = f"""
    <style type='text/css'>
    .map-legend {{
        position: fixed;
        bottom: 20px;
        right: 20px;
        width: {str(arg_width)}px;
        height: auto;
        z-index:9999;
        background-color: rgba(255, 255, 255, 0.9);
        box-shadow: 0 0 15px rgba(0, 0, 0, 0.2);
        padding: 10px;
        font-family: Arial, sans-serif;
        font-size: 14px;
        border-radius: 10px;
    }}
    .legend-title {{
        font-weight: bold;
        margin-bottom: 8px;
    }}
    .legend-labels {{
        list-style: none;
        padding: 0;
        margin: 0;
    }}
    .legend-labels li {{
        list-style: none;
        margin-bottom: 5px;
    }}
"""
    
    # The legend HTML that goes before and after the actual marker entries
    legend_html = f"""
    <div id='map-legend' class='map-legend'>
        <div class='legend-title'>{legend_title}</div>
        <div class='legend-scale'>
        <ul class='legend-labels'>
"""

    legend_html_end = """
    </ul>
        </div>
    </div>
"""
    # Work through the entries to add appropriate html for the circle markers
    for legend_entry in arg_legend_list:
        # Split up legend_entry into the descriptions, classes, etc.
        marker_descrip = legend_entry[0]
        marker_class = legend_entry[1]
        marker_stroke = legend_entry[2]
        stroke_opacity = legend_entry[3]
        marker_fill = legend_entry[4]
        fill_opacity = legend_entry[5]

        # Convert the marker fill color to RGB and add the opacity
        rgb_str = str(ColorHex2RGB(marker_stroke))
        marker_stroke = rgb_str[1:-1] + ', ' + str(stroke_opacity)

        # Convert the marker fill color to RGB and add the opacity
        rgb_str = str(ColorHex2RGB(marker_fill))
        marker_fill = rgb_str[1:-1] + ', ' + str(fill_opacity)

        # Make the appropriate HTML for the legend entry
        entry_now = f"\t\t<li><span class=\'{marker_class}\' style=\'background: rgba({marker_fill}); border: 2px solid rgba({marker_stroke});\'></span>{marker_descrip}</li>\n"
        legend_html = legend_html + entry_now
    
    #print(legend_html)
    # Return the full html
    return legend_html + legend_html_end + basic_CSS + marker_css


In [ ]:
def GenerateHTML4Popup(arg_df_row):
    """Function to generate the HTML to insert into a popup in folium
    AUTHOR:     Mai Tanaka (www.DataDrivenMai.com)
    DATE:       2026-06-30
    REQUIRES: arg_df_row = pandas dataframe row as .itertuples()
    PROMISES: html_popup = html format to pass onto folium html input in popups
    """
    # Determine the color of the Y/N fonts for the data collected
    data_YN = [arg_df_row.rainfall_YN, 
               arg_df_row.temperature_YN, 
               arg_df_row.wind_YN, 
               arg_df_row.sunshine_YN, 
               arg_df_row.relative_humidity_YN, 
               arg_df_row.atmospheric_pressure_YN,
               arg_df_row.snowfall_YN]
    color_YN = []
    for data_YN_now in data_YN:
        if data_YN_now == 'Y':
            color_YN.append('green')
        else:
            color_YN.append('red')
    
    # Make the html_popup (contains a table of meteorological data collected)
    html_popup = f"""
    <h3> {arg_df_row.romaji_name.capitalize()}({arg_df_row.station_name})</h3>
    prec_no: {arg_df_row.prec_no}
    <br>
    block_no: {arg_df_row.block_no}
    <br>
    a or s: {arg_df_row.url_station_type}
    <br>
    Location: {arg_df_row.latitude_decimal:.2f}°, {arg_df_row.longitude_decimal:.2f}°
    <br>
    Elevation: {arg_df_row.elevation} m
    <br>
    <br>    
    <h4>Data Collected</h4>
    <table border="1">
    <tbody>
        <tr>
        <td>rainfall</td>
        <td><span style="color: {color_YN[0]}; font-weight: bold;">{data_YN[0]}</span></td>
        </tr>
        <tf>
        <td>temperature</td>
        <td><span style="color: {color_YN[1]}; font-weight: bold;">{data_YN[1]}</span></td>
        </tr>
        <tr>
        <td>wind direction/speed</td>
        <td><span style="color: {color_YN[2]}; font-weight: bold;">{data_YN[2]}</span></td>
        </tr>
        <tr>
        <td>sunshine</td>
        <td><span style="color: {color_YN[3]}; font-weight: bold;">{data_YN[3]}</span></td>
        </tr>
        <tr>
        <td>relative humidity</td>
        <td><span style="color: {color_YN[4]}; font-weight: bold;">{data_YN[4]}</span></td>
        </tr>
        <tr>
        <td>atmospheric pressure</td>
        <td><span style="color: {color_YN[5]}; font-weight: bold;">{data_YN[5]}</span></td>
        </tr>
        <tr>
        <td>snowfall</td>
        <td><span style="color: {color_YN[6]}; font-weight: bold;">{data_YN[6]}</span></td>
        </tr>
    </tbody>
    </table>
    <br>
    Thermometer height: {arg_df_row.thermometer_height} m
    <br>
    Anemometer height: {arg_df_row.anemometer_height} m
    <br>
    <br>
    <h4>Observation Start Dates</h4>
    Rain: {arg_df_row.observation_start_date_rain}
    <br>
    Other: {arg_df_row.observation_start_date_other}
    """   
    
    return html_popup

In [ ]:
def show_folium_safe(m, height=500):
    """
    Displays a Folium map in a safe IFrame using Base64 encoding.
    This avoids "Trusted" errors, file path issues, and CSS leakage.
    Source: https://github.com/microsoft/vscode-jupyter/issues/17224#issuecomment-3679624559
    """
    # 1. Get the raw HTML string of the map
    html_content = m.get_root().render()
    
    # 2. Encode the HTML to base64
    # This allows us to put the entire map "inside" the URL string
    encoded = base64.b64encode(html_content.encode('utf-8')).decode('utf-8')
    
    # 3. Create a Data URI
    data_uri = f"data:text/html;charset=utf-8;base64,{encoded}"
    
    # 4. Display the IFrame
    # We use width='100%' to fill the cell width, but the CSS is trapped inside
    display(IFrame(src=data_uri, width="100%", height=height))

## {width=40%}

### Top Card 

Read up on [project details and documentation](deets-n-docs.html)



### Bottom 

In [ ]:
#| classes: no-scrollbar
#| title: All Weather Stations

# Default variables
all_opacity = 0.6

# Get the central location for the map
min_lat = min(amedas_df_all['latitude_decimal'])
max_lat = max(amedas_df_all['latitude_decimal'])
mid_lat = min_lat + ((max_lat - min_lat) / 2)
min_long = min(amedas_df_all['longitude_decimal'])
max_long = max(amedas_df_all['longitude_decimal'])
mid_long = min_long + ((max_long - min_long) / 2) + ((max_long - min_long) * 0.1) # Bit of padding added for legend

# Generate folium map of Japan
m = folium.Map(
    location=[mid_lat, mid_long], 
    zoom_start=3, 
    tiles="cartodb positron",
    prefer_canvas=True, 
    width='100%', 
    height='100%'
    )

# Map the JMA stations as circle markers with mouse over information
for rowNow in amedas_df_all.itertuples():
    
    # From the prefecture number, get the jurisdiction name (and thus its marker color)
    juris_now = PrecNo2Name(rowNow.prec_no)    
    color_now = juris_info[juris_now][0]
    
    # Adjust circle marker radius according to size
    if rowNow.url_station_type == 's':
        radius_now = 5
    else:
        radius_now = 3

    # Make the html to display information when mouse hovers over the circle marker
    mouseover_info = f"""
    <h5>{rowNow.romaji_name.capitalize()}({rowNow.station_name})</h5>
    <table border="1" cellpadding="3">
    <tr><td>prec_no</td> <td><b>{rowNow.prec_no}</b></td></tr>
    <tr><td>block_no</td> <td><b>{rowNow.block_no}</b></td></tr>
    <tr><td>a or s</td> <td><b>{rowNow.url_station_type}</b></td></tr>
    </table>
    """
    
    # Add the circle marker to the folium map
    markerNow = folium.CircleMarker(
        location=[rowNow.latitude_decimal, rowNow.longitude_decimal],
        tooltip=mouseover_info,
        color=color_now, 
        stroke=False, 
        fill=True,
        fill_opacity=all_opacity,
        radius=radius_now
    ).add_to(m)

# Create a legend
legend_title = 'Regions'
legend_list = [['Sapporo', 'medium', juris_info['Sapporo'][0], 0, juris_info['Sapporo'][0], all_opacity], 
               ['Sendai', 'medium', juris_info['Sendai'][0], 0, juris_info['Sendai'][0], all_opacity], 
               ['Tokyo', 'medium', juris_info['Tokyo'][0], 0, juris_info['Tokyo'][0], all_opacity], 
               ['Osaka', 'medium', juris_info['Osaka'][0], 0, juris_info['Osaka'][0], all_opacity], 
               ['Fukuoka', 'medium', juris_info['Fukuoka'][0], 0, juris_info['Fukuoka'][0], all_opacity], 
               ['Okinawa', 'medium', juris_info['Okinawa'][0], 0, juris_info['Okinawa'][0], all_opacity], 
               ] # Set stroke opacity to zero as it paints on top of the fill
legend_width_px = 100

# Create the legend_html and add the legend to the map
legend_html = CreateLegend4CircleMarkers(legend_title, legend_width_px, legend_list)
m.get_root().html.add_child(folium.Element(legend_html))

# Fit to bounds (does not seem to work nicely on websites, so map is already centralized upon creation)
#m.fit_bounds([[min_lat, min_long], [max_lat, max_long]], padding_bottom_right=(width_px, 0))

# Show map
#show_folium_safe(m) # Use only when map doesn't render in VSCode
m

## Column {width=60% .tabset .no-scrollbar}


In [ ]:
def MapJurisdiction(juris_now, zoom_now):
    """Function to generate the folium map (m) for the given jurisdiction
    AUTHOR:     Mai Tanaka (www.DataDrivenMai.com)
    DATE:       2026-06-30
    REQUIRES: juris_now = name of jurisdiction to map ('Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa')
              zoom_now = starting zoom level for the map
    PROMISES: m = folium map object of the jurisdiction, complete with legend and popups
    """

    # Draw the jurisdiction information from the juris_info dictionary
    color_now = juris_info[juris_now][0]
    prec_low, prec_high = juris_info[juris_now][1]
    mid_lat, mid_long = juris_info[juris_now][2]

    # Constants
    legend_width_px = 220

    # Generate folium map of the jurisdiction region
    m = folium.Map(
        location=[mid_lat, mid_long], 
        tiles="cartodb positron",
        zoom_start=zoom_now,
        prefer_canvas=True,
        width='100%', 
        height='100%'
    )

    # Build the filtered station list first, then compute numeric bounds
    filtered_rows = [
        rowNow
        for rowNow in amedas_df_all.itertuples()
        if prec_low <= int(rowNow.prec_no) <= prec_high
    ]
    if not filtered_rows:
        raise ValueError(f'No stations found for jurisdiction {juris_now}')

    # Work through each row to plot a circle marker and generate popups 
    for rowNow in filtered_rows:
        # Location of the station
        lat_now = float(rowNow.latitude_decimal)
        long_now = float(rowNow.longitude_decimal)
        
        # Make the mouse over information
        mouseover_info = f"<h5>{rowNow.romaji_name.capitalize()}({rowNow.station_name})</h5>"
        
        # Make the popup information
        html_popup = GenerateHTML4Popup(rowNow)

        # Adjust size of marker and fill opacity according to station type
        if rowNow.station_type == '雨' or rowNow.station_type == '雪':
            radius_now = 2
            fill_opacity = 0.5
        elif rowNow.station_type == '三':
            radius_now = 2
            fill_opacity = 0.2
        elif rowNow.station_type == '四':
            radius_now = 3
            fill_opacity = 0.4
        elif rowNow.station_type == '官':
            radius_now = 5
            fill_opacity = 0.8
            
        # Adjust marker fill color
        if rowNow.snowfall_YN == 'Y':
            stroke_color = '#495057'
            if rowNow.station_type == '雪':
                fill_color = 'white'
            else:
                fill_color = color_now 
        elif rowNow.station_type == '雨':
            fill_color = '#90E0EF'
            stroke_color = '#90E0EF'
        elif rowNow.url_station_type == 's':
            fill_color = color_now
            stroke_color = color_now
        else:
            fill_color = color_now
            stroke_color = color_now
        
        # Make the marker and add it to the folium map
        markerNow = folium.CircleMarker(
            location=[lat_now, long_now],
            tooltip=mouseover_info,
            popup=folium.Popup(html=html_popup,
                        max_width=300, 
                        max_height=150),
            stroke=True,
            weight=2,
            color=stroke_color,
            opacity=1.0,
            radius=radius_now,
            fill_color=fill_color,
            fill_opacity=fill_opacity,        
        ).add_to(m)

    # Create a legend
    legend_title = 'Types of Data Collected'
    legend_list = [['Rainfall', 'small', '#90E0EF', 1.0, '#90E0EF', 0.5], 
                ['Rainfall, temperature, wind', 'small', str(color_now), 1.0, str(color_now), 0.2], 
                ['Rainfall, temperature, wind, relative humidity', 'medium', str(color_now), 1.0, str(color_now), 0.4], 
                ['Rainfall, temperature, wind, relative humidity, sunshine, atmospheric pressure', 'large', str(color_now), 1.0, str(color_now), 0.8], 
                ['Snowfall', 'medium', '#495057', 1.0, '#FFFFFF', 0.5], 
                ]
    legend_html = CreateLegend4CircleMarkers(legend_title, legend_width_px, legend_list)

    # Add legend to map
    m.get_root().html.add_child(folium.Element(legend_html))

    # Fit map to bounds (does not work well with web format, so map is already modified upon creation)
    #m.fit_bounds([[min_lat, min_long], [max_lat, max_long]], padding_bottom_right=(legend_width_px, 0))

    # Return the map
    return m

In [ ]:
#| title: Sapporo

# Designate the jurisdiction and starting zoom level
juris_now = 'Sapporo' # 'Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa'
zoom_now = 6

# Create folium map 
m = MapJurisdiction(juris_now, zoom_now)

# Show map
#show_folium_safe(m)
m

In [ ]:
#| title: Sendai

# Designate the jurisdiction and starting zoom level
juris_now = 'Sendai' # 'Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa'
zoom_now = 6

# Create folium map 
m = MapJurisdiction(juris_now, zoom_now)

# Show map
#show_folium_safe(m)
m

In [ ]:
#| title: Tokyo

# Designate the jurisdiction and starting zoom level
juris_now = 'Tokyo' # 'Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa'
zoom_now = 4

# Create folium map 
m = MapJurisdiction(juris_now, zoom_now)

# Show map
#show_folium_safe(m)
m

In [ ]:
#| title: Osaka

# Designate the jurisdiction and starting zoom level
juris_now = 'Osaka' # 'Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa'
zoom_now = 6

# Create folium map 
m = MapJurisdiction(juris_now, zoom_now)

# Show map
#show_folium_safe(m)
m

In [ ]:
#| title: Fukuoka

# Designate the jurisdiction and starting zoom level
juris_now = 'Fukuoka' # 'Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa'
zoom_now = 5

# Create folium map 
m = MapJurisdiction(juris_now, zoom_now)

# Show map
#show_folium_safe(m)
m

In [ ]:
#| title: Okinawa

# Designate the jurisdiction and starting zoom level
juris_now = 'Okinawa' # 'Sapporo', 'Sendai', 'Tokyo', 'Osaka', 'Fukuoka', 'Okinawa'
zoom_now = 6

# Create folium map 
m = MapJurisdiction(juris_now, zoom_now)

# Show map
#show_folium_safe(m)
m